# Image filtering

In this notebook, we'll explore the basics of image filtering using practical examples. Image filtering consists of modifying the intensity of the pixels in an image with a specific aim, such as:
* removing noise or background
* sharpening an image
* highlighting certain features

This process can aid certain tasks, such as thresholding. However, we should keep in mind that the pixel values in the image are being modified, and therefore, signal intensity should **not** be quantified on filtered images.

## Import libraries

In [ ]:
import numpy as np
from scipy import ndimage
import skimage
import matplotlib.pyplot as plt
import stackview

## Morphological filters

Morphological filters are normally applied to binary masks to perform tasks such as removing small objects, filling holes, and closing edges. The idea is to turn pixels ON and OFF depending on their surroundings, changing whether they belong to the foreground or background class. 

The simplest to understand are the erosion and dilation filters. As their names indicate, these filters either shrink or enlarge ON objects. The size and shape of the filters specify what region is considered around objects to shrink or enlarge them. 

<img src="illustrations/morphological_operations.png" alt="drawing" width="90%" class="center"/>

<sub>Illustration taken from "Introduction to Bioimage Analysis" by Pete Bankhead, [CC-BY 4.0](https://creativecommons.org/licenses/by/4.0/).</sub>

Let's see this in action! We're going to start by creating a binary mask by thresholding an image

In [ ]:
# load an example image
image = skimage.io.imread('images/cells_atlas/50546_727_A8_2.tif')

# subset the array to select the DAPI channel and zoom into a small rectangular region
image_nuclei = image[:700, :, 2]

# threshold with mean algorithm and create binary mask
threshold_otsu = skimage.filters.threshold_mean(image_nuclei)
binary_mask = image_nuclei > threshold_otsu

In [ ]:
plt.imshow(binary_mask, cmap='gray')
plt.title('Binary mask')
plt.axis('off')
plt.show()

### Erosion
We can specify the shape and size of an erosion filter. For example, we can use a circle of radius 2:

In [ ]:
binary_eroded = skimage.morphology.isotropic_erosion(binary_mask, radius=2)

In [ ]:
plt.imshow(binary_eroded, cmap='gray')
plt.title('Binary mask - erosion')
plt.axis('off')
plt.show()

### Dilation
We can specify the shape and size of a dilation filter as well. For example, for a circle of radius 2:

In [ ]:
binary_dilated = skimage.morphology.isotropic_dilation(binary_mask, radius=2)

In [ ]:
plt.imshow(binary_dilated, cmap='gray')
plt.title('Binary mask - dilation')
plt.axis('off')
plt.show()

### Opening and closing
We can run erosion and dilation processes in sequence to achieve the removal of small objects ([**morphological opening**](https://en.wikipedia.org/wiki/Opening_%28morphology%29): first erode, then dilate) or hole filling ([**morphological closing**](https://en.wikipedia.org/wiki/Closing_(morphology)): first dilate, then erode)

In [ ]:
binary_open = skimage.morphology.isotropic_opening(binary_mask, radius=8)

In [ ]:
plt.imshow(binary_open, cmap='gray')
plt.title('Binary mask - opening')
plt.axis('off')
plt.show()

In [ ]:
binary_close = skimage.morphology.isotropic_closing(binary_mask, radius=5)

In [ ]:
plt.imshow(binary_close, cmap='gray')
plt.title('Binary mask - closing')
plt.axis('off')
plt.show()

There's also another useful function in the `scipy` library, which can fill the holes in a binary image. We can see it in action below:

In [ ]:
from scipy import ndimage

binary_fill = ndimage.binary_fill_holes(binary_mask)

In [ ]:
plt.imshow(binary_fill, cmap='gray')
plt.title('Binary mask - fill holes')
plt.axis('off')
plt.show()

## Convolutional filters
Filtering images is at the core of denoising and image processing. The idea is to use a small image that travels across a larger image and does some operation on each sub-region. 

In the simplest case, we can imagine that a small square travels across the image and computes the *local mean* of the pixels with which it overlaps. This is then a *mean* filter.

Multiple properties of the *little travelling square* can change:
- The operation to be run on each sub-region, i.e., the pixel values of the kernel. This can be a local mean, Gaussian, etc.
- The size and shape of the "square"

Filters can be applied to regular images, as well as to masks (binary images) and are often used to denoise or segment images.

`scikit-image` has many filters available under [`skimage.filters`](https://scikit-image.org/docs/stable/api/skimage.filters.html).

<img src="illustrations/convolution.png" alt="drawing" width="50%" class="center"/>

### Mean filter
A mean filter computes the local mean over the kernel as it travels across the image. This has a smoothing effect and can be used for denoising. The size of the kernel determines the amount of smoothing achieved.

Let's load an image and see the filter in action

In [ ]:
image = skimage.io.imread('images/cells_atlas/46658_784_B12_1.tif')[200:450, 250:700, 2]

plt.imshow(image, cmap='gray')
plt.title('Raw image')
plt.axis('off')
plt.show()

In [ ]:
# mean filter radius 2
image_mean2 = skimage.filters.rank.mean(image, skimage.morphology.disk(2))
# mean filter radius 5
image_mean5 = skimage.filters.rank.mean(image, skimage.morphology.disk(5))

# visualise side by side
fig, axs = plt.subplots(1, 3, figsize=(12,6))

axs[0].imshow(image, cmap='gray')
axs[0].set_title('Raw image')
axs[0].axis('off')

axs[1].imshow(image_mean2, cmap='gray')
axs[1].set_title('Mean filter (radius 2)')
axs[1].axis('off')

axs[2].imshow(image_mean5, cmap='gray')
axs[2].set_title('Mean filter (radius 5)')
axs[2].axis('off')

plt.tight_layout()


We can visualise the effect of filtering with a line profile plot

In [ ]:
plt.plot(image[100, 150: 300], label='Raw image')
plt.plot(image_mean2[100, 150: 300], label='Mean filter (radius 2)')
plt.plot(image_mean5[100, 150: 300], label='Mean filter (radius 5)')
plt.legend()

plt.show()

Smoothing can often aid with thresholding, as it blurs some of the smaller features in the image and helps better define object edges.

In [ ]:
# segment raw image
threshold = skimage.filters.threshold_otsu(image)
binary = image > threshold

# segment image with mean filter radius 2 applied
threshold_mean2 = skimage.filters.threshold_otsu(image_mean2)
binary_mean2 = image_mean2 > threshold_mean2

# segment image with mean filter radius 5 applied
threshold_mean5 = skimage.filters.threshold_otsu(image_mean5)
binary_mean5 = image_mean5 > threshold_mean5

# visualise side by side
fig, axs = plt.subplots(1, 3, figsize=(12,6))

axs[0].imshow(binary, cmap='gray')
axs[0].set_title('Binary mask (raw image)')
axs[0].axis('off')

axs[1].imshow(binary_mean2, cmap='gray')
axs[1].set_title('Binary mask (mean filter radius 2)')
axs[1].axis('off')

axs[2].imshow(binary_mean5, cmap='gray')
axs[2].set_title('Binary mask (mean filter radius 5)')
axs[2].axis('off')

plt.tight_layout()

### Gaussian blur
A similar denoising/smoothing effect can be achieved with a [Gaussian filter](https://en.wikipedia.org/wiki/Gaussian_blur). In this case, the kernel has higher values in the centre and lower values on the edges, with a characteristic bell shape. This filter is widely used in microscopy, as it matches the shape of optical aberrations of lenses in a light microscope.

<img src="illustrations/gaussian_kernel.png" alt="drawing" width="50%" class="center"/>

<sub>Adapted from (Shipitko et al, 2018)</sub>

In [ ]:
# gaussian blur sigma 2
image_gauss2 = skimage.filters.gaussian(image, sigma=2, preserve_range=True)
# gaussian blur sigma 5
image_gauss5 = skimage.filters.gaussian(image, sigma=5, preserve_range=True)

# segment image with gaussian blur sigma 2 applied
threshold_gauss2 = skimage.filters.threshold_otsu(image_gauss2)
binary_gauss2 = image_gauss2 > threshold_gauss2
# segment image with gaussian blur sigma 5 applied
threshold_gauss5 = skimage.filters.threshold_otsu(image_gauss5)
binary_gauss5 = image_gauss5 > threshold_gauss5

# visualise side by side
fig, axs = plt.subplots(2, 3, figsize=(12,6))

axs[0,0].imshow(image, cmap='gray')
axs[0,0].set_title('Raw image')
axs[0,0].axis('off')

axs[0,1].imshow(image_gauss2, cmap='gray')
axs[0,1].set_title('Gaussian blur (sigma 2)')
axs[0,1].axis('off')

axs[0,2].imshow(image_gauss5, cmap='gray')
axs[0,2].set_title('Gaussian blur (sigma 5)')
axs[0,2].axis('off')

axs[1,0].imshow(binary, cmap='gray')
axs[1,0].set_title('Binary mask (raw image)')
axs[1,0].axis('off')

axs[1,1].imshow(binary_gauss2, cmap='gray')
axs[1,1].set_title('Binary mask (Gaussian blur sigma 2)')
axs[1,1].axis('off')

axs[1,2].imshow(binary_gauss5, cmap='gray')
axs[1,2].set_title('Binary mask (Gaussian blur sigma 5)')
axs[1,2].axis('off')

plt.tight_layout()


In [ ]:
plt.plot(image[100, 150: 300], label='Raw image')
plt.plot(image_gauss2[100, 150: 300], label='Gaussian blur (sigma 2)')
plt.plot(image_gauss5[100, 150: 300], label='Gaussian blur (sigma 5)')
plt.legend()

plt.show()

### Edge detection
Filtering can facilitate different tasks. Another useful one in image processing is the detection of object edges. The idea in this case is for the kernel to resemble an edge (e.g., a vertical or horizontal bright line with darker pixels on either side), and we are looking for parts of the images that match the kernel shape. One example in this family is the [Sobel filter](https://en.wikipedia.org/wiki/Sobel_operator).

In [ ]:
# sobel vertical
image_sobel_v = skimage.filters.sobel_v(image)
# sobel horizontal
image_sobel_h = skimage.filters.sobel_h(image)
# sobel
image_sobel = skimage.filters.sobel(image)

# visualise side by side
fig, axs = plt.subplots(1, 4, figsize=(14,6))

axs[0].imshow(image, cmap='gray')
axs[0].set_title('Raw image')
axs[0].axis('off')

axs[1].imshow(image_sobel_v, cmap='gray')
axs[1].set_title('Sobel filter (vertical)')
axs[1].axis('off')

axs[2].imshow(image_sobel_h, cmap='gray')
axs[2].set_title('Sobel filter (horizontal)')
axs[2].axis('off')

axs[3].imshow(image_sobel, cmap='gray')
axs[3].set_title('Sobel filter')
axs[3].axis('off')

plt.tight_layout()

In [ ]:
image = skimage.io.imread('https://cildata.crbs.ucsd.edu/media/images/13901/13901.tif')

In [ ]:
stackview.curtain(image, image > 600, zoom_factor=0.5)

In [ ]:
from skimage import filters
def preprocess(im):
    f = im
    f = f.astype(float) - filters.rank.minimum(f, footprint=np.ones((200,200), dtype='bool'))
    f = f * (f > 0)
    f = f / filters.gaussian(f, sigma=50)
    f = np.clip(f, 0, 4)
    return f

image_preprocessed = preprocess(image)

In [ ]:
print('Original vs preprocessed')
stackview.curtain(image, image_preprocessed, zoom_factor=0.5)

In [ ]:
print('Original vs preprocessed and tresholded')

stackview.curtain(image, image_preprocessed>1.2, zoom_factor=0.5)

### Applications in deep learning

Similarly to detecting edges in images, convolutional filters are used in deep learning to detect all types of patterns in images.

In this process, the convolutional filter is learned from the data, rather than being pre-defined.

[![IMAGE ALT TEXT](http://img.youtube.com/vi/VUkFo6IXMJc/0.jpg)](http://www.youtube.com/watch?v=VUkFo6IXMJc "Video Title")

## Non-linear filters

Another class of filters does not use convolution to filter the image but does some other type of operation on small neighbourhoods of the image. 

### Median filter

For example, it can calculate the median value of each sub-region in the image. This is extremely useful if our image has pixels that are "dead" or have extremely large values (salt-and-pepper noise) or if we need to clean object edges. We can choose a "footprint" (the shape and size of the neighbourhood) and apply the median filter to the image.

<img src="illustrations/median_filter.png" alt="drawing" width="40%" class="center"/>

<sub>Illustration adapted from https://www.southampton.ac.uk/~msn/book/new_demo/median/</sub>


In [ ]:
# median filter radius 2
image_median2 = skimage.filters.rank.median(image, skimage.morphology.disk(2))
# median filter radius 5
image_median5 = skimage.filters.rank.median(image, skimage.morphology.disk(5))

# visualise side by side
fig, axs = plt.subplots(1, 3, figsize=(12,6))

axs[0].imshow(image, cmap='gray')
axs[0].set_title('Raw image')
axs[0].axis('off')

axs[1].imshow(image_median2, cmap='gray')
axs[1].set_title('Median filter (radius 2)')
axs[1].axis('off')

axs[2].imshow(image_median5, cmap='gray')
axs[2].set_title('Median filter (radius 5)')
axs[2].axis('off')

plt.tight_layout()


In [ ]:
plt.plot(image[100, 150: 300], label='Raw image')
plt.plot(image_median2[100, 150: 300], label='Median filter (radius 2)')
plt.plot(image_median5[100, 150: 300], label='Median filter (radius 5)')
plt.legend()

plt.show()

## Test your understanding
Let's test your understanding with `exercise_02_filtering.ipynb`.